# Amazon EMR 8.1 completes support for the Apache Iceberg v3 data types

**A worked example: default values, the unknown type, geospatial types, and nanosecond timestamps**

**With [Amazon EMR](https://aws.amazon.com/emr/) 8.1, the [Apache Iceberg](https://iceberg.apache.org/) [v3 data types](https://iceberg.apache.org/spec/) are
complete.** Default column values, the unknown type, the GEOMETRY and GEOGRAPHY geospatial types,
and nanosecond-precision timestamps join the VARIANT type (covered in a separate post) —
every v3 type, now available with [Apache Spark](https://spark.apache.org/) on Amazon EMR.

This notebook accompanies the AWS blog post of the same name. It exercises five industry-neutral
Apache Iceberg v3 data-type capabilities on Amazon EMR 8.1 — default values, an unknown-type
placeholder, planar and spherical geospatial types, and nanosecond timestamps. These solve
modeling problems common to telecom, IoT, financial services, logistics, healthcare, and
scientific computing. To keep things concrete, we ground them in one running example: a small set
of related retail tables that reference each other. Retail is just the setting; wherever you see
orders, stores, or clickstream, read your own domain.

The example is five related retail tables, linked by customer and store keys:

| Table | Iceberg v3 capability | Role in the example |
|---|---|---|
| `orders` | Default column values | Add `fulfillment_channel` to existing orders with no rewrite |
| `customers` | The unknown type (`VOID`) | Reserve `loyalty_tier` as a schema contract before launch |
| `delivery_zones` | `GEOMETRY` | Delivery-area polygons; point-in-polygon lookups |
| `stores` | `GEOGRAPHY` | Canonical store location on a round earth |
| `clickstream` | Nanosecond timestamps | Order high-volume events that tie at microsecond precision |

At the end, a single query joins `customers`, `delivery_zones`, and `stores` to find which store
delivers to each customer. Run this notebook on an [Amazon EMR Serverless](https://aws.amazon.com/emr/serverless/)
interactive application or an [EMR Studio](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-studio.html) notebook attached to an
**[Amazon EMR 8.1](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-810-release.html)** cluster.

## Prerequisites

- An **Amazon EMR 8.1** environment (this notebook was validated on [Amazon EMR Serverless](https://aws.amazon.com/emr/serverless/)).
- An **[Amazon S3](https://aws.amazon.com/s3/)** bucket for the Iceberg warehouse.
- Permission to use the **[AWS Glue Data Catalog](https://docs.aws.amazon.com/glue/latest/dg/catalog-and-crawler.html)**.

Replace `amzn-s3-demo-bucket` with your own bucket before running.

## Configure the Spark session

Configure the Iceberg session extensions and a Glue-backed catalog named `glue_catalog`. To use
the geospatial types (`GEOMETRY` and `GEOGRAPHY`) you must also enable geospatial support, which
is **off by default**.

> If your kernel already provides a `spark` session (as EMR Studio does), set these keys through
> the kernel's session configuration instead of building a new `SparkSession`.

In [ ]:
from pyspark.sql import SparkSession

WAREHOUSE = "s3://amzn-s3-demo-bucket/iceberg-v3-features/"  # <-- your bucket

spark = (
    SparkSession.builder
    .appName("iceberg_v3_lakehouse")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.glue_catalog",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue_catalog.catalog-impl",
            "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue_catalog.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.glue_catalog.warehouse", WAREHOUSE)
    .config("spark.sql.geospatial.enabled", "true")   # required for GEOMETRY / GEOGRAPHY
    .getOrCreate()
)

spark.sql("CREATE DATABASE IF NOT EXISTS glue_catalog.retail_lakehouse_v3")

The one requirement that ties every example together: create your tables with **format version 3**.
The v3 types are rejected on a v2 table, so every `CREATE TABLE` below sets
`TBLPROPERTIES ('format-version'='3')`.

## 1. Default column values — the `orders` table

Adding a column to a table that already holds data is one of the most common schema changes in any
domain — and historically one of the most expensive: on billions of rows across terabytes of
Parquet, materializing it the old way means rewriting every file, which can take hours. In Iceberg
v3 a column can instead carry a **default** in table metadata: existing rows read it back with **no
data-file rewrite**, and new rows that omit the column also get it.

In our example, the `orders` table has years of history and the team now wants a
`fulfillment_channel` on every order.

> We default to `'unspecified'` — the historical orders predate the concept, so their true channel
> was never recorded and we should not assert otherwise.

In [ ]:
T = "glue_catalog.retail_lakehouse_v3.orders"
spark.sql(f"DROP TABLE IF EXISTS {T}")

spark.sql(f"""
    CREATE TABLE {T} (
        order_id INT, customer_id INT, store_id INT, order_total DECIMAL(10,2)
    ) USING iceberg TBLPROPERTIES ('format-version'='3')
""")

# historical orders, written before the new column existed
spark.sql(f"INSERT INTO {T} VALUES (1001, 9001, 1, 89.99), (1002, 9002, 2, 15.50), (1003, 9001, 1, 240.00)")

# add the column with a default AFTER rows already exist
spark.sql(f"ALTER TABLE {T} ADD COLUMN fulfillment_channel STRING DEFAULT 'unspecified'")

# a brand-new order can still set the channel explicitly
spark.sql(f"INSERT INTO {T} VALUES (1004, 9003, 2, 59.00, 'ship_to_home')")

spark.sql(f"""
    SELECT order_id, customer_id, store_id, order_total, fulfillment_channel
    FROM {T} ORDER BY order_id
""").show(truncate=False)

**What you'll see:** the pre-existing orders (1001-1003) read back
`fulfillment_channel = 'unspecified'` — the backfilled default — while the new order (1004) keeps
its explicit `'ship_to_home'`. None of the original data files were rewritten.

## 2. The unknown type (`VOID`) — the `customers` table

Every team eventually needs to publish a column before it knows what will go in it — a field
reserved for a roadmap feature, or one a downstream producer will populate later. The v3 **unknown
type** models exactly this: an always-null column whose eventual type is intentionally left open,
so consumers can be built against a stable **schema contract** today.

In our example, the team is planning a loyalty program for next quarter but hasn't finalized how
tiers will be represented, so it publishes `loyalty_tier` now; the column reads `NULL` until the
program launches.

> In Spark SQL the unknown type is spelled **`VOID`** (`UNKNOWN` raises a parser error). The table
> also carries a `home_location` point we use in the geospatial join later.

In [ ]:
T = "glue_catalog.retail_lakehouse_v3.customers"
spark.sql(f"DROP TABLE IF EXISTS {T}")

spark.sql(f"""
    CREATE TABLE {T} (
        customer_id INT, email STRING, loyalty_tier VOID, home_location GEOMETRY(4326)
    ) USING iceberg TBLPROPERTIES ('format-version'='3')
""")

spark.sql(f"""
    INSERT INTO {T} VALUES
    (9001, 'ava@example.com',  NULL, st_setsrid(st_geomfromtext('POINT(-122.33 47.61)'), 4326)),
    (9002, 'ben@example.com',  NULL, st_setsrid(st_geomfromtext('POINT(-73.98 40.75)'),  4326)),
    (9003, 'cara@example.com', NULL, st_setsrid(st_geomfromtext('POINT(-87.63 41.88)'),  4326))
""")

spark.sql(f"DESCRIBE {T}").show(truncate=False)

**What you'll see:** `loyalty_tier` reads `NULL` for every row, and `DESCRIBE` reports its data
type as `void` (with `home_location` typed `geometry(4326)`).

## 3. `GEOMETRY` — the `delivery_zones` table

Shapes on a plane — service areas, coverage cells, field boundaries, floor plans, routes — turn up
across logistics, telecom, agriculture, and gaming. The v3 **`GEOMETRY`** type stores planar
(Cartesian) values — points, lines, polygons — with a spatial reference identifier (SRID). Over a
bounded area, planar math is accurate enough and fast, and — importantly — `GEOMETRY` is the type
the EMR 8.1 spatial functions actually compute on.

In our example, each store serves a delivery area modeled as a polygon.

> A numeric SRID is required (`GEOMETRY(4326)`) — a named CRS such as `'OGC:CRS84'` is not
> accepted. Build from WKT with `st_geomfromtext(...)` (or WKB with `st_geomfromwkb(unhex('...'))`)
> and stamp the SRID with `st_setsrid(..., 4326)` — a bare constructor yields SRID 0, which will not
> cast into a `GEOMETRY(4326)` column.

In [ ]:
T = "glue_catalog.retail_lakehouse_v3.delivery_zones"
spark.sql(f"DROP TABLE IF EXISTS {T}")

spark.sql(f"""
    CREATE TABLE {T} (
        store_id INT, zone GEOMETRY(4326)
    ) USING iceberg TBLPROPERTIES ('format-version'='3')
""")

spark.sql(f"""
    INSERT INTO {T} VALUES
    (1, st_setsrid(st_geomfromtext('POLYGON((-122.4 47.5, -122.2 47.5, -122.2 47.7, -122.4 47.7, -122.4 47.5))'), 4326)),
    (2, st_setsrid(st_geomfromtext('POLYGON((-74.1 40.6, -73.9 40.6, -73.9 40.8, -74.1 40.8, -74.1 40.6))'), 4326))
""")

spark.sql(f"""
    SELECT store_id, st_srid(zone) AS srid, round(st_area(zone), 4) AS area_sq_deg
    FROM {T} ORDER BY store_id
""").show(truncate=False)

**What you'll see:** each polygon stores and reads back from the `GEOMETRY(4326)` column, with the
SRID preserved and `st_area` returning the planar area. We put these zones to work in the
point-in-polygon query at the end.

## 4. `GEOGRAPHY` — the `stores` table

When points span the globe — fleets, sensors, shipments, weather stations — a flat plane no longer
models distance correctly. The v3 **`GEOGRAPHY`** type interprets coordinates on a spherical earth
model, making it the portable, correct system of record for a point on the earth.

In our example, a store's canonical location is stored as a `GEOGRAPHY` value.

> On EMR 8.1 the `ST_*` **computation** functions operate on `GEOMETRY`, not `GEOGRAPHY`: `st_x`,
> `st_y`, `st_distance`, and `st_within` all reject a geography argument, and `st_astext` is not
> registered. Read a geography value with `st_srid(...)` and `hex(st_asbinary(...))`, and project
> to `GEOMETRY` when you need spatial computation (as the delivery zones do).

In [ ]:
T = "glue_catalog.retail_lakehouse_v3.stores"
spark.sql(f"DROP TABLE IF EXISTS {T}")

spark.sql(f"""
    CREATE TABLE {T} (
        store_id INT, city STRING, geo_location GEOGRAPHY(4326)
    ) USING iceberg TBLPROPERTIES ('format-version'='3')
""")

# POINT(lon lat) for each store, as little-endian WKB
spark.sql(f"""
    INSERT INTO {T} VALUES
    (1, 'Seattle',  st_setsrid(st_geogfromwkb(unhex('010100000085EB51B81E955EC0AE47E17A14CE4740')), 4326)),
    (2, 'New York', st_setsrid(st_geogfromwkb(unhex('010100000000000000008052C07B14AE47E15A4440')), 4326))
""")

spark.sql(f"""
    SELECT store_id, city, st_srid(geo_location) AS srid,
           hex(st_asbinary(geo_location)) AS wkb
    FROM {T} ORDER BY store_id
""").show(truncate=False)

**What you'll see:** each store's value round-trips exactly — `st_srid` returns `4326` and the WKB
read back matches the WKB written byte for byte.

## 5. Nanosecond timestamps — the `clickstream` table

Iceberg v2 timestamps carry microsecond (6-digit) precision. A high-volume clickstream collector
ingests millions of events per second from many sessions at once, and at that rate events from
different sessions routinely arrive within the same microsecond. If the ingestion timestamp only
resolves to microseconds, those events tie and their true arrival order is lost — which matters for
correct sessionization, deduplication, and ordered replay. Iceberg v3 preserves the full resolution
with **nanosecond (9-digit)** timestamps. On EMR 8.1, declare one with precision 9:
`TIMESTAMP_NTZ(9)` (no time zone) or `TIMESTAMP(9)`.

In our example, three clickstream events from different sessions are ingested inside the **same
microsecond**. At microsecond precision they collapse to one instant; the nanosecond arrival
timestamp gives them a deterministic total order.

> There is no nanosecond literal — `CAST` a string to the nanosecond type.

In [ ]:
T = "glue_catalog.retail_lakehouse_v3.clickstream"
spark.sql(f"DROP TABLE IF EXISTS {T}")

spark.sql(f"""
    CREATE TABLE {T} (
        event_id INT, session_id STRING, event_type STRING, event_ts TIMESTAMP_NTZ(9)
    ) USING iceberg TBLPROPERTIES ('format-version'='3')
""")

# three events from different sessions, all ingested inside the SAME microsecond
spark.sql(f"""
    INSERT INTO {T} VALUES
    (1, 'sess-a', 'page_view',   CAST('2026-09-07 12:34:56.000000001' AS TIMESTAMP_NTZ(9))),
    (2, 'sess-b', 'add_to_cart', CAST('2026-09-07 12:34:56.000000042' AS TIMESTAMP_NTZ(9))),
    (3, 'sess-c', 'checkout',    CAST('2026-09-07 12:34:56.000000999' AS TIMESTAMP_NTZ(9)))
""")

# ordering by the nanosecond timestamp gives a deterministic total order
spark.sql(f"""
    SELECT session_id, event_type, CAST(event_ts AS STRING) AS event_ts_ns,
           substring(CAST(event_ts AS STRING), 1, 26) AS truncated_to_micro
    FROM {T} ORDER BY event_ts
""").show(truncate=False)

**What you'll see:** the `truncated_to_micro` column shows all three events as the identical instant
`2026-09-07 12:34:56.000000` — a three-way tie at microsecond precision. The full nanosecond values
differ, so ordering by `event_ts` returns them in their true arrival order. All nine fractional
digits survive the round-trip with no truncation.

## Bringing it together: which store delivers to each customer

The tables are connected, so we can answer a real question in one query: for each customer, which
store's delivery zone contains their home? This joins `customers` to `delivery_zones` with a
point-in-polygon test, then to `stores` for the label. Both operands are `GEOMETRY`, so the test
uses `st_within`.

> On EMR 8.1, `st_within(point, polygon)` and `st_intersects(a, b)` are available, but `st_contains`
> is not registered — use `st_within` with the arguments in point-then-polygon order.

In [ ]:
spark.sql(f"""
    SELECT c.customer_id, c.email, s.store_id, s.city AS serving_store
    FROM glue_catalog.retail_lakehouse_v3.customers c
    JOIN glue_catalog.retail_lakehouse_v3.delivery_zones z ON st_within(c.home_location, z.zone)
    JOIN glue_catalog.retail_lakehouse_v3.stores s ON s.store_id = z.store_id
    ORDER BY c.customer_id
""").show(truncate=False)

# LEFT JOIN surfaces customers with no delivery coverage (e.g. 9003, Chicago)
spark.sql(f"""
    SELECT c.customer_id, c.email,
           CASE WHEN z.store_id IS NULL THEN 'no delivery zone'
                ELSE CONCAT('store ', z.store_id) END AS coverage
    FROM glue_catalog.retail_lakehouse_v3.customers c
    LEFT JOIN glue_catalog.retail_lakehouse_v3.delivery_zones z ON st_within(c.home_location, z.zone)
    ORDER BY c.customer_id
""").show(truncate=False)

**What you'll see:** customer 9001 (Seattle) maps to store 1 and 9002 (New York) to store 2; 9003
(Chicago) falls in no zone and shows `no delivery zone` in the LEFT JOIN. One query exercises three
of the v3 tables at once, with the geospatial types doing real work.

## Clean up

To avoid ongoing charges, drop the tables and database you created, then delete any warehouse
objects left in Amazon S3.

In [ ]:
for tbl in ["orders", "customers", "delivery_zones", "stores", "clickstream"]:
    spark.sql(f"DROP TABLE IF EXISTS glue_catalog.retail_lakehouse_v3.{tbl}")
spark.sql("DROP DATABASE IF EXISTS glue_catalog.retail_lakehouse_v3")